In [1]:
import pandas as pd

games = pd.read_csv(
    "nba_games_raw.csv",
    dtype={"GAME_ID": str}
)

games.head()

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON_STR
0,41983,1.610613e+09,BOS,Boston Celtics,0048300079,1984-06-12,BOS vs. LAL,W,240,111,...,20.0,32.0,52.0,18,11.0,3,14,23,NaN,1983-84
1,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300079,1984-06-12,LAL @ BOS,L,240,102,...,9.0,24.0,33.0,28,9.0,8,17,32,NaN,1983-84
2,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300078,1984-06-10,LAL vs. BOS,W,240,119,...,19.0,25.0,44.0,31,9.0,3,14,28,NaN,1983-84
3,41983,1.610613e+09,BOS,Boston Celtics,0048300078,1984-06-10,BOS @ LAL,L,240,108,...,13.0,28.0,41.0,25,7.0,8,19,24,NaN,1983-84
4,41983,1.610613e+09,BOS,Boston Celtics,0048300077,1984-06-08,BOS vs. LAL,W,240,121,...,13.0,38.0,51.0,28,8.0,5,18,32,NaN,1983-84


In [2]:
games["GAME_TYPE"] = games["GAME_ID"].str[:3]

games["GAME_TYPE"].value_counts()

,count
GAME_TYPE,
002,98234
004,6746
001,4053
003,194
005,74
006,6


In [3]:
playoff_games = games[
    games["GAME_TYPE"] == "004"
].copy()

In [4]:
playoff_games["SEASON_STR"].value_counts().sort_index()

,count
SEASON_STR,
1983-84,158
1984-85,136
1985-86,136
1986-87,142
1987-88,160
1988-89,124
1989-90,144
1990-91,136
1991-92,146


In [5]:
print(playoff_games.shape)

(6746, 30)


In [6]:
playoff_games["IS_HOME"] = (
    playoff_games["MATCHUP"]
    .str.contains("vs.")
    .astype(int)
)

playoff_games["WIN"] = (
    playoff_games["WL"] == "W"
).astype(int)

playoff_games["LOSS"] = (
    playoff_games["WL"] == "L"
).astype(int)

In [7]:
[
    "SEASON_STR",
    "TEAM_ID",
    "TEAM_NAME"
]

['SEASON_STR', 'TEAM_ID', 'TEAM_NAME']

In [8]:
playoff_profiles = (
    playoff_games
    .groupby(
        ["SEASON_STR", "TEAM_ID", "TEAM_NAME"]
    )
    .agg(
        games=("GAME_ID", "count"),

        wins=("WIN", "sum"),
        losses=("LOSS", "sum"),

        ppg=("PTS", "mean"),

        fg_pct=("FG_PCT", "mean"),
        fg3_pct=("FG3_PCT", "mean"),
        ft_pct=("FT_PCT", "mean"),

        oreb=("OREB", "mean"),
        dreb=("DREB", "mean"),
        reb=("REB", "mean"),

        ast=("AST", "mean"),
        stl=("STL", "mean"),
        blk=("BLK", "mean"),
        tov=("TOV", "mean"),

        plus_minus=("PLUS_MINUS", "mean")
    )
    .reset_index()
)

playoff_profiles["win_pct"] = (
    playoff_profiles["wins"]
    /
    playoff_profiles["games"]
)

playoff_profiles.head()

,SEASON_STR,TEAM_ID,TEAM_NAME,games,wins,losses,ppg,fg_pct,fg3_pct,ft_pct,oreb,dreb,reb,ast,stl,blk,tov,plus_minus,win_pct
0,1983-84,1.610613e+09,Atlanta Hawks,5,2,3,93.600000,0.422400,0.266600,0.794600,13.600000,24.400000,38.000000,20.600000,7.800000,4.600000,15.600000,NaN,0.400000
1,1983-84,1.610613e+09,Boston Celtics,23,15,8,110.956522,0.474522,0.351182,0.782348,15.478261,28.391304,43.869565,23.043478,8.565217,5.347826,15.695652,NaN,0.652174
2,1983-84,1.610613e+09,Dallas Mavericks,10,4,6,101.700000,0.440200,0.098778,0.835800,15.900000,27.300000,43.200000,21.700000,5.700000,3.600000,13.000000,NaN,0.400000
3,1983-84,1.610613e+09,Denver Nuggets,5,2,3,121.800000,0.506600,0.242800,0.851000,13.200000,28.800000,42.000000,27.000000,8.400000,4.600000,15.000000,NaN,0.400000
4,1983-84,1.610613e+09,Los Angeles Lakers,21,14,7,117.190476,0.534524,0.278053,0.718571,13.142857,29.857143,43.000000,30.285714,8.238095,6.809524,15.904762,NaN,0.666667


In [9]:
home_stats = (
    playoff_games[
        playoff_games["IS_HOME"] == 1
    ]
    .groupby(
        ["SEASON_STR","TEAM_ID"]
    )
    .agg(
        home_games=("GAME_ID","count"),
        home_wins=("WIN","sum"),
        home_losses=("LOSS","sum")
    )
    .reset_index()
)

home_stats["home_win_pct"] = (
    home_stats["home_wins"]
    /
    home_stats["home_games"]
)

In [10]:
away_stats = (
    playoff_games[
        playoff_games["IS_HOME"] == 0
    ]
    .groupby(
        ["SEASON_STR","TEAM_ID"]
    )
    .agg(
        away_games=("GAME_ID","count"),
        away_wins=("WIN","sum"),
        away_losses=("LOSS","sum")
    )
    .reset_index()
)

away_stats["away_win_pct"] = (
    away_stats["away_wins"]
    /
    away_stats["away_games"]
)

In [11]:
playoff_profiles = playoff_profiles.merge(
    home_stats,
    on=["SEASON_STR","TEAM_ID"],
    how="left"
)

playoff_profiles = playoff_profiles.merge(
    away_stats,
    on=["SEASON_STR","TEAM_ID"],
    how="left"
)

In [12]:
game_pairs = playoff_games[
    [
        "GAME_ID",
        "TEAM_ID",
        "PTS",
        "FG_PCT",
        "FG3_PCT"
    ]
].copy()

opp = game_pairs.rename(
    columns={
        "TEAM_ID":"OPP_TEAM_ID",
        "PTS":"OPP_PTS",
        "FG_PCT":"OPP_FG_PCT",
        "FG3_PCT":"OPP_FG3_PCT"
    }
)

merged = game_pairs.merge(
    opp,
    on="GAME_ID"
)

merged = merged[
    merged["TEAM_ID"]
    !=
    merged["OPP_TEAM_ID"]
]

In [13]:
opp_stats = (
    merged
    .groupby("TEAM_ID")
    .agg(
        opp_ppg=("OPP_PTS","mean"),
        opp_fg_pct=("OPP_FG_PCT","mean"),
        opp_fg3_pct=("OPP_FG3_PCT","mean")
    )
    .reset_index()
)

playoff_profiles = playoff_profiles.merge(
    opp_stats,
    on="TEAM_ID",
    how="left"
)

In [14]:
playoff_profiles["point_diff"] = (
    playoff_profiles["ppg"]
    -
    playoff_profiles["opp_ppg"]
)

playoff_profiles["fg_pct_diff"] = (
    playoff_profiles["fg_pct"]
    -
    playoff_profiles["opp_fg_pct"]
)

playoff_profiles["fg3_pct_diff"] = (
    playoff_profiles["fg3_pct"]
    -
    playoff_profiles["opp_fg3_pct"]
)

playoff_profiles["assist_turnover_ratio"] = (
    playoff_profiles["ast"]
    /
    playoff_profiles["tov"]
)

playoff_profiles["home_away_gap"] = (
    playoff_profiles["home_win_pct"]
    -
    playoff_profiles["away_win_pct"]
)

In [15]:
game_pairs = playoff_games[
    [
        "GAME_ID",
        "TEAM_ID",
        "REB",
        "AST",
        "TOV"
    ]
].copy()

opp = game_pairs.rename(
    columns={
        "TEAM_ID":"OPP_TEAM_ID",
        "REB":"OPP_REB",
        "AST":"OPP_AST",
        "TOV":"OPP_TOV"
    }
)

merged = game_pairs.merge(
    opp,
    on="GAME_ID"
)

merged = merged[
    merged["TEAM_ID"]
    !=
    merged["OPP_TEAM_ID"]
]

In [16]:
opp_box = (
    merged
    .groupby("TEAM_ID")
    .agg(
        opp_reb=("OPP_REB","mean"),
        opp_ast=("OPP_AST","mean"),
        opp_tov=("OPP_TOV","mean")
    )
    .reset_index()
)

playoff_profiles = playoff_profiles.merge(
    opp_box,
    on="TEAM_ID",
    how="left"
)

In [17]:
playoff_profiles["reb_diff"] = (
    playoff_profiles["reb"]
    -
    playoff_profiles["opp_reb"]
)

playoff_profiles["ast_diff"] = (
    playoff_profiles["ast"]
    -
    playoff_profiles["opp_ast"]
)

playoff_profiles["tov_diff"] = (
    playoff_profiles["opp_tov"]
    -
    playoff_profiles["tov"]
)

In [18]:
playoff_profiles["fgm"] = (
    playoff_games
    .groupby(["SEASON_STR","TEAM_ID"])
    ["FGM"]
    .mean()
    .values
)

playoff_profiles["fga"] = (
    playoff_games
    .groupby(["SEASON_STR","TEAM_ID"])
    ["FGA"]
    .mean()
    .values
)

playoff_profiles["fg3m"] = (
    playoff_games
    .groupby(["SEASON_STR","TEAM_ID"])
    ["FG3M"]
    .mean()
    .values
)

playoff_profiles["fta"] = (
    playoff_games
    .groupby(["SEASON_STR","TEAM_ID"])
    ["FTA"]
    .mean()
    .values
)

playoff_profiles["ftm"] = (
    playoff_games
    .groupby(["SEASON_STR","TEAM_ID"])
    ["FTM"]
    .mean()
    .values
)

In [19]:
playoff_profiles["poss"] = (
    playoff_profiles["fga"]
    +
    0.44 * playoff_profiles["fta"]
    -
    playoff_profiles["oreb"]
    +
    playoff_profiles["tov"]
)

In [20]:
playoff_profiles["pace"] = (
    playoff_profiles["poss"]
)

In [21]:
playoff_profiles["efg_pct"] = (
    (
        playoff_profiles["fgm"]
        +
        0.5 * playoff_profiles["fg3m"]
    )
    /
    playoff_profiles["fga"]
)

In [22]:
playoff_profiles["ts_pct"] = (
    playoff_profiles["ppg"]
    /
    (
        2 *
        (
            playoff_profiles["fga"]
            +
            0.44 * playoff_profiles["fta"]
        )
    )
)

In [23]:
playoff_profiles["ortg"] = (
    100 *
    playoff_profiles["ppg"]
    /
    playoff_profiles["poss"]
)

playoff_profiles["drtg"] = (
    100 *
    playoff_profiles["opp_ppg"]
    /
    playoff_profiles["poss"]
)

playoff_profiles["net_rating"] = (
    playoff_profiles["ortg"]
    -
    playoff_profiles["drtg"]
)

In [24]:
team_profiles_v3 = pd.read_csv("team_profiles_v3.csv")

print(set(team_profiles_v3.columns)
==
set(playoff_profiles.columns))

False


In [25]:
print("Columns in playoff_profiles:")
print(playoff_profiles.columns)

print("\nColumns in team_profiles_v3:")
print(team_profiles_v3.columns)

Columns in playoff_profiles:
Index(['SEASON_STR', 'TEAM_ID', 'TEAM_NAME', 'games', 'wins', 'losses', 'ppg',
       'fg_pct', 'fg3_pct', 'ft_pct', 'oreb', 'dreb', 'reb', 'ast', 'stl',
       'blk', 'tov', 'plus_minus', 'win_pct', 'home_games', 'home_wins',
       'home_losses', 'home_win_pct', 'away_games', 'away_wins', 'away_losses',
       'away_win_pct', 'opp_ppg', 'opp_fg_pct', 'opp_fg3_pct', 'point_diff',
       'fg_pct_diff', 'fg3_pct_diff', 'assist_turnover_ratio', 'home_away_gap',
       'opp_reb', 'opp_ast', 'opp_tov', 'reb_diff', 'ast_diff', 'tov_diff',
       'fgm', 'fga', 'fg3m', 'fta', 'ftm', 'poss', 'pace', 'efg_pct', 'ts_pct',
       'ortg', 'drtg', 'net_rating'],
      dtype='object')

Columns in team_profiles_v3:
Index(['SEASON_STR', 'TEAM_ID', 'TEAM_NAME', 'games', 'wins', 'losses', 'ppg',
       'fg_pct', 'fg3_pct', 'ft_pct', 'oreb', 'dreb', 'reb', 'ast', 'stl',
       'blk', 'tov', 'plus_minus', 'win_pct', 'home_games', 'home_wins',
       'home_losses', 'home_win_pc

In [26]:
playoff_profiles["playoff_team"] = 1

In [27]:
playoff_profiles["TEAM_KEY"] = (
    playoff_profiles["SEASON_STR"]
    + "_"
    + playoff_profiles["TEAM_NAME"]
)

In [28]:
print(
    list(playoff_profiles.columns)
    ==
    list(team_profiles_v3.columns)
)

False


In [29]:
playoff_profiles = playoff_profiles[team_profiles_v3.columns]

In [30]:
print(
    list(playoff_profiles.columns)
    ==
    list(team_profiles_v3.columns)
)

True


In [31]:
print(
    list(playoff_profiles.columns)
    ==
    list(team_profiles_v3.columns)
)

True


In [32]:
playoff_profiles.to_csv(
    "playoff_team_profiles_v1.csv",
    index=False
)

In [33]:
matchups = pd.read_csv(
    "matchups_v3.csv",
    dtype={"GAME_ID": str}
)

/tmp/ipykernel_4243/1636711592.py:1: DtypeWarning: Columns (29,60) have mixed types. Specify dtype option on import or set low_memory=False.
  matchups = pd.read_csv(


In [34]:
matchups.shape

(8300, 192)

In [35]:
matchups.head()

,SEASON_ID_HOME,TEAM_ID_HOME,TEAM_ABBREVIATION_HOME,TEAM_NAME_HOME,GAME_ID,GAME_DATE_HOME,MATCHUP_HOME,WL_HOME,MIN_HOME,PTS_HOME,...,AWAY_drtg,AWAY_net_rating,AWAY_efg_pct,AWAY_ts_pct,pace_matchup_diff,ortg_matchup_diff,drtg_matchup_diff,net_rating_matchup_diff,efg_pct_matchup_diff,ts_pct_matchup_diff
0,41983,1.610613e+09,BOS,Boston Celtics,0048300079,1984-06-12,BOS vs. LAL,W,240.0,111.0,...,105.141128,4.099141,0.536360,0.574260,-2.289294,-1.469943,-3.104243,1.634299,-0.038306,-0.023226
1,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300078,1984-06-10,LAL vs. BOS,W,240.0,119.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226
2,41983,1.610613e+09,BOS,Boston Celtics,0048300077,1984-06-08,BOS vs. LAL,W,240.0,121.0,...,105.141128,4.099141,0.536360,0.574260,-2.289294,-1.469943,-3.104243,1.634299,-0.038306,-0.023226
3,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300076,1984-06-06,LAL vs. BOS,L,265.0,125.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226
4,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300075,1984-06-03,LAL vs. BOS,W,240.0,137.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226


In [36]:
matchups["GAME_ID"] = (
    matchups["GAME_ID"]
    .astype(str)
    .str.zfill(10)
)

In [37]:
playoff_matchups = matchups[
    matchups["GAME_ID"].str.startswith("004")
].copy()

print(playoff_matchups.shape)

(568, 192)


In [38]:
matchups["season_year"].min()


1983.0

In [39]:
matchups["season_year"].max()

1990.0

In [40]:
matchups.head()

,SEASON_ID_HOME,TEAM_ID_HOME,TEAM_ABBREVIATION_HOME,TEAM_NAME_HOME,GAME_ID,GAME_DATE_HOME,MATCHUP_HOME,WL_HOME,MIN_HOME,PTS_HOME,...,AWAY_drtg,AWAY_net_rating,AWAY_efg_pct,AWAY_ts_pct,pace_matchup_diff,ortg_matchup_diff,drtg_matchup_diff,net_rating_matchup_diff,efg_pct_matchup_diff,ts_pct_matchup_diff
0,41983,1.610613e+09,BOS,Boston Celtics,0048300079,1984-06-12,BOS vs. LAL,W,240.0,111.0,...,105.141128,4.099141,0.536360,0.574260,-2.289294,-1.469943,-3.104243,1.634299,-0.038306,-0.023226
1,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300078,1984-06-10,LAL vs. BOS,W,240.0,119.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226
2,41983,1.610613e+09,BOS,Boston Celtics,0048300077,1984-06-08,BOS vs. LAL,W,240.0,121.0,...,105.141128,4.099141,0.536360,0.574260,-2.289294,-1.469943,-3.104243,1.634299,-0.038306,-0.023226
3,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300076,1984-06-06,LAL vs. BOS,L,265.0,125.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226
4,41983,1.610613e+09,LAL,Los Angeles Lakers,0048300075,1984-06-03,LAL vs. BOS,W,240.0,137.0,...,102.036885,5.733440,0.498054,0.551034,2.289294,1.469943,3.104243,-1.634299,0.038306,0.023226


In [41]:
playoff_matchups.to_csv(
    "playoff_matchups_v3.csv",
    index=False
)

In [42]:
playoff_train = playoff_matchups[
    playoff_matchups["season_year"] <= 2018
].copy()

playoff_valid = playoff_matchups[
    (playoff_matchups["season_year"] >= 2019)
    &
    (playoff_matchups["season_year"] <= 2022)
].copy()

playoff_test = playoff_matchups[
    playoff_matchups["season_year"] >= 2023
].copy()

print("Train:", playoff_train.shape)
print("Valid:", playoff_valid.shape)
print("Test :", playoff_test.shape)

Train: (568, 192)
Valid: (0, 192)
Test : (0, 192)


In [43]:
playoff_train.to_csv(
    "playoff_train_full.csv",
    index=False
)

playoff_valid.to_csv(
    "playoff_valid_full.csv",
    index=False
)

playoff_test.to_csv(
    "playoff_test_full.csv",
    index=False
)

In [44]:
clean_cols = [

    "win_pct_matchup_diff",
    "home_win_pct_matchup_diff",
    "away_win_pct_matchup_diff",

    "ppg_matchup_diff",
    "opp_ppg_matchup_diff",

    "fg_pct_matchup_diff",
    "opp_fg_pct_matchup_diff",

    "fg3_pct_matchup_diff",
    "opp_fg3_pct_matchup_diff",

    "ft_pct_matchup_diff",

    "reb_matchup_diff",
    "ast_matchup_diff",
    "stl_matchup_diff",
    "blk_matchup_diff",
    "tov_matchup_diff",

    "point_diff_matchup_diff",

    "assist_turnover_ratio_matchup_diff",
    "home_away_gap_matchup_diff",

    "reb_diff_matchup_diff",
    "ast_diff_matchup_diff",
    "tov_diff_matchup_diff",

    "pace_matchup_diff",
    "ortg_matchup_diff",
    "drtg_matchup_diff",
    "net_rating_matchup_diff",
    "efg_pct_matchup_diff",
    "ts_pct_matchup_diff",

    "TARGET"
]

In [45]:
playoff_train_clean = playoff_train[
    clean_cols
].copy()

playoff_valid_clean = playoff_valid[
    clean_cols
].copy()

playoff_test_clean = playoff_test[
    clean_cols
].copy()

In [46]:
playoff_train_clean.to_csv(
    "playoff_train_clean.csv",
    index=False
)

playoff_valid_clean.to_csv(
    "playoff_valid_clean.csv",
    index=False
)

playoff_test_clean.to_csv(
    "playoff_test_clean.csv",
    index=False
)

In [47]:
basketball_features = [

    "fg_pct_matchup_diff",
    "opp_fg_pct_matchup_diff",

    "fg3_pct_matchup_diff",
    "opp_fg3_pct_matchup_diff",

    "ft_pct_matchup_diff",

    "reb_matchup_diff",
    "ast_matchup_diff",
    "stl_matchup_diff",
    "blk_matchup_diff",
    "tov_matchup_diff",

    "assist_turnover_ratio_matchup_diff",

    "reb_diff_matchup_diff",
    "ast_diff_matchup_diff",
    "tov_diff_matchup_diff",

    "pace_matchup_diff",
    "ortg_matchup_diff",
    "drtg_matchup_diff",
    "efg_pct_matchup_diff",
    "ts_pct_matchup_diff"
]

In [48]:
X_train = playoff_train[basketball_features]
y_train = playoff_train["TARGET"]

X_valid = playoff_valid[basketball_features]
y_valid = playoff_valid["TARGET"]

X_test = playoff_test[basketball_features]
y_test = playoff_test["TARGET"]

print(X_train.shape)
print(X_valid.shape)
print(X_test.shape)

(568, 19)
(0, 19)
(0, 19)


In [49]:
from xgboost import XGBClassifier

xgb_playoff = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,

    subsample=0.8,
    colsample_bytree=0.8,

    random_state=42,
    eval_metric="logloss"
)

xgb_playoff.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

/usr/local/lib/python3.12/dist-packages/xgboost/callback.py:265: UserWarning: [14:00:30] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:56: Empty dataset at worker: 0
  score: str = model.eval_set(evals, epoch, self.metric, self._output_margin)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [50]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss
)

valid_prob = xgb_playoff.predict_proba(X_valid)[:,1]
valid_pred = (valid_prob > 0.5).astype(int)

print("Validation Accuracy:",
      accuracy_score(y_valid, valid_pred))

print("Validation ROC-AUC:",
      roc_auc_score(y_valid, valid_prob))

print("Validation LogLoss:",
      log_loss(y_valid, valid_prob))

Validation Accuracy: nan


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [51]:
test_prob = xgb_playoff.predict_proba(X_test)[:,1]
test_pred = (test_prob > 0.5).astype(int)

print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

print("Test ROC-AUC:",
      roc_auc_score(y_test, test_prob))

print("Test LogLoss:",
      log_loss(y_test, test_prob))

Test Accuracy: nan


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [52]:
import pandas as pd

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_playoff.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
11,reb_diff_matchup_diff,0.078845
15,ortg_matchup_diff,0.066046
18,ts_pct_matchup_diff,0.065249
0,fg_pct_matchup_diff,0.061435
16,drtg_matchup_diff,0.055429
5,reb_matchup_diff,0.055378
14,pace_matchup_diff,0.055299
17,efg_pct_matchup_diff,0.054184
1,opp_fg_pct_matchup_diff,0.052419
12,ast_diff_matchup_diff,0.050743


In [53]:
all_features = [

    "win_pct_matchup_diff",
    "home_win_pct_matchup_diff",
    "away_win_pct_matchup_diff",

    "ppg_matchup_diff",
    "opp_ppg_matchup_diff",

    "fg_pct_matchup_diff",
    "opp_fg_pct_matchup_diff",

    "fg3_pct_matchup_diff",
    "opp_fg3_pct_matchup_diff",

    "ft_pct_matchup_diff",

    "reb_matchup_diff",
    "ast_matchup_diff",
    "stl_matchup_diff",
    "blk_matchup_diff",
    "tov_matchup_diff",

    "point_diff_matchup_diff",

    "assist_turnover_ratio_matchup_diff",
    "home_away_gap_matchup_diff",

    "reb_diff_matchup_diff",
    "ast_diff_matchup_diff",
    "tov_diff_matchup_diff",

    "pace_matchup_diff",
    "ortg_matchup_diff",
    "drtg_matchup_diff",
    "net_rating_matchup_diff",
    "efg_pct_matchup_diff",
    "ts_pct_matchup_diff"
]

In [54]:
X_train = playoff_train[all_features]
y_train = playoff_train["TARGET"]

X_valid = playoff_valid[all_features]
y_valid = playoff_valid["TARGET"]

X_test = playoff_test[all_features]
y_test = playoff_test["TARGET"]

In [55]:
print("Train")
print(y_train.value_counts(normalize=True))

print("\nValid")
print(y_valid.value_counts(normalize=True))

print("\nTest")
print(y_test.value_counts(normalize=True))

Train
TARGET
1.0    0.672535
0.0    0.327465
Name: proportion, dtype: float64

Valid
Series([], Name: proportion, dtype: float64)

Test
Series([], Name: proportion, dtype: float64)


In [56]:
from xgboost import XGBClassifier

xgb_playoff_all = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_playoff_all.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [57]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss
)

valid_prob = xgb_playoff_all.predict_proba(X_valid)[:,1]
valid_pred = (valid_prob > 0.5).astype(int)

print("Validation Accuracy:",
      accuracy_score(y_valid, valid_pred))

print("Validation ROC-AUC:",
      roc_auc_score(y_valid, valid_prob))

print("Validation LogLoss:",
      log_loss(y_valid, valid_prob))

Validation Accuracy: nan


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [58]:
test_prob = xgb_playoff_all.predict_proba(X_test)[:,1]
test_pred = (test_prob > 0.5).astype(int)

print("Test Accuracy:",
      accuracy_score(y_test, test_pred))

print("Test ROC-AUC:",
      roc_auc_score(y_test, test_prob))

print("Test LogLoss:",
      log_loss(y_test, test_prob))

Test Accuracy: nan


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [59]:
playoff_games = pd.read_csv(
    "nba_games_raw.csv",
    dtype={"GAME_ID": str}
)

playoff_games["GAME_TYPE"] = (
    playoff_games["GAME_ID"].str[:3]
)

playoff_games = playoff_games[
    playoff_games["GAME_TYPE"] == "004"
].copy()

In [60]:
home = playoff_games[
    playoff_games["MATCHUP"]
    .str.contains("vs.")
].copy()

away = playoff_games[
    playoff_games["MATCHUP"]
    .str.contains("@")
].copy()

In [61]:
playoff_matchups = home.merge(
    away,
    on="GAME_ID",
    suffixes=("_HOME", "_AWAY")
)

In [62]:
playoff_matchups["TARGET"] = (
    playoff_matchups["WL_HOME"] == "W"
).astype(int)

In [63]:
playoff_matchups["HOME_TEAM_KEY"] = (
    playoff_matchups["SEASON_STR_HOME"]
    + "_"
    + playoff_matchups["TEAM_NAME_HOME"]
)

playoff_matchups["AWAY_TEAM_KEY"] = (
    playoff_matchups["SEASON_STR_AWAY"]
    + "_"
    + playoff_matchups["TEAM_NAME_AWAY"]
)

In [64]:
home_profiles = playoff_profiles.add_prefix(
    "HOME_"
)

playoff_matchups = playoff_matchups.merge(
    home_profiles,
    left_on="HOME_TEAM_KEY",
    right_on="HOME_TEAM_KEY",
    how="left"
)

In [65]:
away_profiles = playoff_profiles.add_prefix(
    "AWAY_"
)

playoff_matchups = playoff_matchups.merge(
    away_profiles,
    left_on="AWAY_TEAM_KEY",
    right_on="AWAY_TEAM_KEY",
    how="left"
)

In [66]:
diff_features = [

    "win_pct",
    "home_win_pct",
    "away_win_pct",

    "ppg",
    "opp_ppg",

    "fg_pct",
    "opp_fg_pct",

    "fg3_pct",
    "opp_fg3_pct",

    "ft_pct",

    "reb",
    "ast",
    "stl",
    "blk",
    "tov",

    "point_diff",

    "assist_turnover_ratio",
    "home_away_gap",

    "reb_diff",
    "ast_diff",
    "tov_diff",

    "pace",
    "ortg",
    "drtg",
    "net_rating",
    "efg_pct",
    "ts_pct"
]

for feat in diff_features:

    playoff_matchups[f"{feat}_matchup_diff"] = (
        playoff_matchups[f"HOME_{feat}"]
        -
        playoff_matchups[f"AWAY_{feat}"]
    )

In [67]:
playoff_matchups["season_year"] = (
    playoff_matchups["SEASON_STR_HOME"]
    .str[:4]
    .astype(int)
)

In [68]:
matchup_cols = [
    c for c in playoff_matchups.columns
    if c.endswith("_matchup_diff")
]

print(len(matchup_cols))
print(matchup_cols)

27
['win_pct_matchup_diff', 'home_win_pct_matchup_diff', 'away_win_pct_matchup_diff', 'ppg_matchup_diff', 'opp_ppg_matchup_diff', 'fg_pct_matchup_diff', 'opp_fg_pct_matchup_diff', 'fg3_pct_matchup_diff', 'opp_fg3_pct_matchup_diff', 'ft_pct_matchup_diff', 'reb_matchup_diff', 'ast_matchup_diff', 'stl_matchup_diff', 'blk_matchup_diff', 'tov_matchup_diff', 'point_diff_matchup_diff', 'assist_turnover_ratio_matchup_diff', 'home_away_gap_matchup_diff', 'reb_diff_matchup_diff', 'ast_diff_matchup_diff', 'tov_diff_matchup_diff', 'pace_matchup_diff', 'ortg_matchup_diff', 'drtg_matchup_diff', 'net_rating_matchup_diff', 'efg_pct_matchup_diff', 'ts_pct_matchup_diff']


In [69]:
playoff_matchups.to_csv(
    "playoff_matchups_v4.csv",
    index=False
)

print(playoff_matchups.shape)

(3373, 198)


In [70]:
playoff_matchups[
    [
        "HOME_games",
        "AWAY_games",
        "HOME_win_pct",
        "AWAY_win_pct"
    ]
].head()

,HOME_games,AWAY_games,HOME_win_pct,AWAY_win_pct
0,23,21,0.652174,0.666667
1,21,23,0.666667,0.652174
2,23,21,0.652174,0.666667
3,21,23,0.666667,0.652174
4,21,23,0.666667,0.652174


In [71]:
playoff_train_v2 = playoff_matchups[
    playoff_matchups["season_year"] <= 2022
].copy()

playoff_test_v2 = playoff_matchups[
    playoff_matchups["season_year"] >= 2023
].copy()

print(playoff_train_v2.shape)
print(playoff_test_v2.shape)

(3124, 198)
(249, 198)


In [72]:
playoff_train_v2.to_csv(
    "playoff_train_v4_full.csv",
    index=False
)

playoff_test_v2.to_csv(
    "playoff_test_v4_full.csv",
    index=False
)

In [73]:
all_features = [

    "win_pct_matchup_diff",
    "home_win_pct_matchup_diff",
    "away_win_pct_matchup_diff",

    "ppg_matchup_diff",
    "opp_ppg_matchup_diff",

    "fg_pct_matchup_diff",
    "opp_fg_pct_matchup_diff",

    "fg3_pct_matchup_diff",
    "opp_fg3_pct_matchup_diff",

    "ft_pct_matchup_diff",

    "reb_matchup_diff",
    "ast_matchup_diff",
    "stl_matchup_diff",
    "blk_matchup_diff",
    "tov_matchup_diff",

    "point_diff_matchup_diff",

    "assist_turnover_ratio_matchup_diff",
    "home_away_gap_matchup_diff",

    "reb_diff_matchup_diff",
    "ast_diff_matchup_diff",
    "tov_diff_matchup_diff",

    "pace_matchup_diff",
    "ortg_matchup_diff",
    "drtg_matchup_diff",
    "net_rating_matchup_diff",
    "efg_pct_matchup_diff",
    "ts_pct_matchup_diff"
]

In [74]:
clean_cols = all_features + ["TARGET"]

playoff_train_v2_clean = playoff_train_v2[
    clean_cols
].copy()

playoff_test_v2_clean = playoff_test_v2[
    clean_cols
].copy()

In [75]:
playoff_train_v2_clean.to_csv(
    "playoff_train_v4_clean.csv",
    index=False
)

playoff_test_v2_clean.to_csv(
    "playoff_test_v4_clean.csv",
    index=False
)

In [76]:
from xgboost import XGBClassifier

X_train = playoff_train_v2[basketball_features]
y_train = playoff_train_v2["TARGET"]

X_test = playoff_test_v2[basketball_features]
y_test = playoff_test_v2["TARGET"]

xgb_basketball = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_basketball.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [77]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

prob = xgb_basketball.predict_proba(X_test)[:,1]
pred = (prob > 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, prob))
print("LogLoss  :", log_loss(y_test, prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

Accuracy : 0.6666666666666666
ROC-AUC  : 0.6983019613005133
LogLoss  : 0.664658704415186

Confusion Matrix
[[ 48  59]
 [ 24 118]]


In [78]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_basketball.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
).head(20)

,feature,importance
15,ortg_matchup_diff,0.153355
18,ts_pct_matchup_diff,0.067876
11,reb_diff_matchup_diff,0.055563
5,reb_matchup_diff,0.055066
17,efg_pct_matchup_diff,0.051592
10,assist_turnover_ratio_matchup_diff,0.048973
7,stl_matchup_diff,0.048779
14,pace_matchup_diff,0.048486
0,fg_pct_matchup_diff,0.045893
1,opp_fg_pct_matchup_diff,0.044474


In [79]:
X_train = playoff_train_v2[all_features]
y_train = playoff_train_v2["TARGET"]

X_test = playoff_test_v2[all_features]
y_test = playoff_test_v2["TARGET"]

xgb_all = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_all.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [80]:
import joblib

playoff_bundle = {
    "model": xgb_basketball,
    "features": basketball_features,
}

joblib.dump(
    playoff_bundle,
    "nba_playoff_simulator_2.pkl"
)

print("Playoff model saved.")

Playoff model saved.


In [ ]:
prob = xgb_all.predict_proba(X_test)[:,1]
pred = (prob > 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, pred))
print("ROC-AUC  :", roc_auc_score(y_test, prob))
print("LogLoss  :", log_loss(y_test, prob))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, pred))

In [ ]:
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_all.feature_importances_
})

importance.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss
)

# Create specific X_test for each model
X_test_basketball = playoff_test_v2[basketball_features]
X_test_all = playoff_test_v2[all_features]

# Re-calculate metrics for xgb_basketball model
prob_basketball = xgb_basketball.predict_proba(X_test_basketball)[:,1]
pred_basketball = (prob_basketball > 0.5).astype(int)

acc_basketball = accuracy_score(y_test, pred_basketball)
auc_basketball = roc_auc_score(y_test, prob_basketball)
ll_basketball = log_loss(y_test, prob_basketball)

# Re-calculate metrics for xgb_all model
prob_all = xgb_all.predict_proba(X_test_all)[:,1]
pred_all = (prob_all > 0.5).astype(int)

acc_all = accuracy_score(y_test, pred_all)
auc_all = roc_auc_score(y_test, prob_all)
ll_all = log_loss(y_test, prob_all)

results = pd.DataFrame({
    "Model": [
        "Playoff Basketball Features",
        "Playoff All Features"
    ],
    "Accuracy": [
        acc_basketball,
        acc_all
    ],
    "ROC_AUC": [
        auc_basketball,
        auc_all
    ],
    "LogLoss": [
        ll_basketball,
        ll_all
    ]
})

results

In [ ]:
test_prob = xgb_all.predict_proba(X_test)[:,1]
test_pred = (test_prob > 0.5).astype(int)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    test_pred
)

print(cm)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

plt.imshow(cm)

plt.colorbar()

plt.xticks([0,1])
plt.yticks([0,1])

plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center"
        )

plt.title("Playoff Model Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        test_pred
    )
)

In [ ]:
from sklearn.metrics import brier_score_loss

brier = brier_score_loss(
    y_test,
    test_prob
)

print(
    "Brier Score:",
    brier
)

In [ ]:
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_test,
    test_prob,
    n_bins=10
)

In [ ]:
plt.figure(figsize=(8,6))

plt.plot(
    prob_pred,
    prob_true,
    marker="o",
    label="Playoff XGB"
)

plt.plot(
    [0,1],
    [0,1],
    "--",
    label="Perfect Calibration"
)

plt.xlabel("Predicted Probability")
plt.ylabel("Actual Win Rate")

plt.title(
    "Playoff Model Calibration Curve"
)

plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,6))

plt.hist(
    test_prob,
    bins=20
)

plt.xlabel(
    "Predicted Home Win Probability"
)

plt.ylabel(
    "Count"
)

plt.title(
    "Playoff Prediction Distribution"
)

plt.show()

In [ ]:
from sklearn.metrics import accuracy_score

for threshold in [
    0.40,
    0.45,
    0.50,
    0.55,
    0.60
]:

    pred = (
        test_prob > threshold
    ).astype(int)

    acc = accuracy_score(
        y_test,
        pred
    )

    print(
        f"Threshold={threshold:.2f} "
        f"Accuracy={acc:.4f}"
    )

In [ ]:
import joblib

playoff_bundle = {
    "model": xgb_all,
    "features": all_features,
}

joblib.dump(
    playoff_bundle,
    "nba_playoff_simulator.pkl"
)

print("Playoff model saved.")